[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milioe/casos-ia-ibero-diplomado/blob/main/modulo_4/08_Word2Vec_scaling.ipynb)


# Word2Vec a escala — un corpus en español

En el notebook **07** vimos la idea de Word2Vec (a mano con Keras y con un corpus pequeño). Aquí hacemos el salto práctico con **`gensim`**, la librería que implementa Word2Vec de verdad (CBOW, skip-gram, negative sampling).

Cargamos **todos** los fragmentos `spanishText_*` del corpus Kaggle (se descargan solos desde Drive), entrenamos CBOW y Skip-gram, exploramos vecinos, analogías, suma/resta de vectores y PCA 3D.

**La enseñanza central (The Bitter Lesson):** por muy limpio que limpiemos el texto, lo que más mueve la calidad es el **volumen de datos**. Para entender tu corpus a fondo, generar texto largo o comparar PDFs enteros, más adelante veremos **transformers** (BERT, GPT).


## Setup — instalar librerías

Necesitamos:

- `gensim` — implementación eficiente de Word2Vec
- `scikit-learn` — PCA para visualización
- `plotly` — gráficos 3D interactivos


In [1]:
%pip install -q numpy matplotlib pandas scikit-learn gensim plotly gdown

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import re
import zipfile
import random
import pickle
from pathlib import Path
from collections import Counter

import gdown

import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from gensim.models import Word2Vec
from sklearn.decomposition import PCA

np.random.seed(42)
random.seed(42)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})

# Stopwords en español (misma lista didáctica que en 02-PDF_reporte.ipynb)
STOP = {
    "el", "la", "los", "las", "un", "una", "unos", "unas", "y", "o", "u", "de", "del", "al",
    "en", "que", "con", "por", "para", "como", "se", "su", "sus", "lo", "le", "les", "a",
    "no", "si", "ya", "más", "menos", "tan", "entre", "sobre", "ser", "es", "son", "fue",
    "ha", "han", "he", "hay", "está", "están", "este", "esta", "esto", "ese", "esa", "eso",
}


def similitud_coseno(u, v):
    return float(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v) + 1e-9))


def vecinos_de_vector(modelo, vec, topn=8, excluir=None):
    excluir = set(excluir or [])
    sims = []
    for w in modelo.wv.index_to_key:
        if w in excluir:
            continue
        sims.append((w, similitud_coseno(modelo.wv[w], vec)))
    sims.sort(key=lambda x: -x[1])
    return sims[:topn]


## Verificar archivos

Los fragmentos `spanishText_*` deben estar en `corpus_descargado/`. La celda de descarga los baja de [Google Drive](https://drive.google.com/drive/folders/1QRkbYOqrlKsxEk8eB94pyZ82aO0OUX2v?usp=sharing) (un solo `archive.zip` de ~490 MB) y los descomprime ahí.

In [ ]:
BASE_DIR = Path(".")
CORPUS_DIR = BASE_DIR / "corpus_descargado"
CORPUS_DIR.mkdir(exist_ok=True)

def listar_textos_corpus():
    """Fragmentos spanishText_* ya descomprimidos en corpus_descargado/."""
    return sorted(p for p in CORPUS_DIR.glob("spanishText_*") if p.is_file())

textos = listar_textos_corpus()
print(f"Fragmentos del corpus encontrados: {len(textos)}")
for t in textos[:5]:
    print(f"  - {t.name}")
if len(textos) > 5:
    print(f"  ... y {len(textos) - 5} más")

# Corpus completo en español (Kaggle)

Aquí entrenamos con **los fragmentos del [corpus de 120M palabras en español](https://www.kaggle.com/datasets/rtatman/120-million-word-spanish-corpus/data)** (57 archivos de texto, subidos a [Drive](https://drive.google.com/drive/folders/1QRkbYOqrlKsxEk8eB94pyZ82aO0OUX2v?usp=sharing) en un solo `archive.zip`).

Descarga, carga, entrenamiento y exploración viven en este notebook.

**Más texto → mejores embeddings** (The Bitter Lesson).

## The Bitter Lesson

Richard Sutton (investigador de RL) resume una idea que en NLP se ve clarísimo:

> **No ganan los trucos de limpieza más ingeniosos, sino los métodos simples con MUCHOS datos y MUCHO cómputo.**

Aquí: limpiar líneas ayuda, pero el salto real viene del **volumen**: con pocos textos los vecinos salen ruidosos; con **cientos de miles de oraciones** del corpus español completo, las relaciones empiezan a tener sentido.


## Descargar y descomprimir el corpus

`gdown` baja `archive.zip` de la carpeta pública de [Google Drive](https://drive.google.com/drive/folders/1QRkbYOqrlKsxEk8eB94pyZ82aO0OUX2v?usp=sharing) y descomprimimos los fragmentos en `corpus_descargado/`. Si ya están ahí, no se vuelve a bajar nada.

Para demo rápida: `N_ARCHIVOS = 3`. Para todo el corpus: `N_ARCHIVOS = None` (necesita bastante RAM).

In [ ]:
DRIVE_FOLDER_ID = "1QRkbYOqrlKsxEk8eB94pyZ82aO0OUX2v"

N_ARCHIVOS = None  # None = los 57 fragmentos; ej. 3 para demo rápida

def descargar_corpus(n_archivos=None):
    ya = listar_textos_corpus()
    if ya and (n_archivos is None or len(ya) >= n_archivos):
        print(f"Ya tienes {len(ya)} fragmentos, no descargo nada.")
        return

    zips = list(CORPUS_DIR.rglob("archive.zip"))
    if not zips:
        print("Descargando archive.zip (~490 MB)...")
        gdown.download_folder(id=DRIVE_FOLDER_ID, output=str(CORPUS_DIR), quiet=True)
        zips = list(CORPUS_DIR.rglob("archive.zip"))
    if not zips:
        raise FileNotFoundError("No se pudo bajar archive.zip de Drive. Descárgalo a mano y ponlo en corpus_descargado/.")

    with zipfile.ZipFile(zips[0]) as zf:
        # solo los spanishText_* de la raíz del zip (ignora carpetas duplicadas)
        nombres = sorted(n for n in zf.namelist() if n.startswith("spanishText_") and "/" not in n)
        if n_archivos is not None:
            nombres = nombres[:n_archivos]
        for n in nombres:
            if not (CORPUS_DIR / n).exists():
                zf.extract(n, CORPUS_DIR)
    print(f"Listo: {len(nombres)} fragmentos en {CORPUS_DIR}/")

descargar_corpus(N_ARCHIVOS)

archivos_texto = listar_textos_corpus()
if N_ARCHIVOS is not None:
    archivos_texto = archivos_texto[:N_ARCHIVOS]
print(f"\nTotal de fragmentos a procesar: {len(archivos_texto)}")

## Leer y tokenizar **todos** los fragmentos

Cada fragmento es un archivo de texto (formato XML por líneas). Saltamos etiquetas y `ENDOFARTICLE`. Limpieza: minúsculas, solo letras españolas, oraciones con al menos `MIN_TOKENS` palabras.

`MAX_LINEAS_POR_ARCHIVO = None` lee **cada archivo completo** (corpus entero). Para una demo corta en laptop, pon por ejemplo `80_000`.

In [ ]:
MIN_TOKENS = 5
MAX_LINEAS_POR_ARCHIVO = None  # None = archivo completo; ej. 80_000 para prueba rápida

def _linea_a_oracion(linea):
    linea = linea.strip().lower()
    if not linea or linea.startswith("<") or "endofarticle" in linea:
        return None
    linea = re.sub(r"[^a-záéíóúüñ ]+", " ", linea)
    linea = re.sub(r"\s+", " ", linea).strip()
    if not linea:
        return None
    tokens = linea.split()
    return tokens if len(tokens) >= MIN_TOKENS else None

def cargar_corpus(archivos, max_lineas_por_archivo=MAX_LINEAS_POR_ARCHIVO):
    oraciones = []
    for i, ruta in enumerate(archivos, 1):
        print(f"[{i}/{len(archivos)}] {ruta.name}")
        with open(ruta, "rb") as f:
            for j, raw in enumerate(f):
                if max_lineas_por_archivo is not None and j >= max_lineas_por_archivo:
                    break
                tokens = _linea_a_oracion(raw.decode("utf-8", errors="ignore"))
                if tokens:
                    oraciones.append(tokens)
        print(f"             {len(oraciones):,} oraciones acumuladas")
    return oraciones


In [ ]:
if not archivos_texto:
    raise FileNotFoundError("No hay fragmentos del corpus. Corre la celda de descarga de arriba.")

oraciones_es = cargar_corpus(archivos_texto)
print(f"\nOraciones cargadas (total): {len(oraciones_es):,}")
print(f"Tokens aproximados: {sum(len(o) for o in oraciones_es):,}")
print("\nEjemplo:")
print(oraciones_es[100][:20])

## Entrenar CBOW y Skip-gram

Entrenamos dos modelos:

- `sg=0` → **CBOW** (contexto → palabra central)
- `sg=1` → **Skip-gram** (palabra central → contexto)

`min_count=10` descarta palabras muy raras (aparecen menos de 10 veces).


In [20]:
VECTOR_SIZE = 100
WINDOW = 5
MIN_COUNT = 10
EPOCHS = 5

print("Entrenando CBOW...")
es_cbow = Word2Vec(
    sentences=oraciones_es,
    vector_size=VECTOR_SIZE,
    window=WINDOW,
    min_count=MIN_COUNT,
    workers=6,
    sg=0,
    epochs=EPOCHS,
    seed=42
)

print("\nEntrenando Skip-gram...")
es_sg = Word2Vec(
    sentences=oraciones_es,
    vector_size=VECTOR_SIZE,
    window=WINDOW,
    min_count=MIN_COUNT,
    workers=6,
    sg=1,
    epochs=EPOCHS,
    seed=42
)

print(f"\nVocab CBOW: {len(es_cbow.wv)}")
print(f"Vocab Skip-gram: {len(es_sg.wv)}")

Entrenando CBOW...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl


Entrenando Skip-gram...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl


Vocab CBOW: 178309
Vocab Skip-gram: 178309


## Guardar modelos en disco

Tras entrenar, los modelos viven en memoria (`es_cbow`, `es_sg`). Para usarlos después (o con `08_word2vec_cli.py`), guárdalos aquí en la **misma carpeta del módulo**:

- `modelo_word2vec_cbow.pkl`
- `modelo_word2vec_sg.pkl`

Pon `GUARDAR_MODELOS = True` y ejecuta la celda de abajo **una vez** al terminar el entrenamiento.

In [21]:
GUARDAR_MODELOS = True  # False si no quieres escribir los .pkl en disco

if GUARDAR_MODELOS:
    ruta_cbow = BASE_DIR / "modelo_word2vec_cbow.pkl"
    ruta_sg = BASE_DIR / "modelo_word2vec_sg.pkl"
    with open(ruta_cbow, "wb") as f:
        pickle.dump(es_cbow, f)
    with open(ruta_sg, "wb") as f:
        pickle.dump(es_sg, f)
    print(f"Guardado CBOW:     {ruta_cbow.resolve()}")
    print(f"Guardado Skip-gram: {ruta_sg.resolve()}")
else:
    print("GUARDAR_MODELOS=False — solo en memoria (es_cbow, es_sg)")

Guardado CBOW:     /Users/emiliosandoval/Documents/ibero/casos-ia-ibero-diplomado/Modulo 4 - NLP/modelo_word2vec_cbow.pkl
Guardado Skip-gram: /Users/emiliosandoval/Documents/ibero/casos-ia-ibero-diplomado/Modulo 4 - NLP/modelo_word2vec_sg.pkl


## Explorar vocabulario del corpus

Veamos qué palabras tiene el modelo entrenado con el corpus grande.


In [22]:
vocab_corpus = list(es_sg.wv.index_to_key)
print(f"Total de palabras en vocabulario: {len(vocab_corpus)}")
print(f"\nPrimeras 60 palabras:")
print(vocab_corpus[:60])

Total de palabras en vocabulario: 178309

Primeras 60 palabras:
['de', 'la', 'en', 'el', 'y', 'a', 'que', 'los', 'del', 'se', 'un', 'por', 'con', 'las', 'una', 'su', 'es', 'al', 'como', 'para', 'fue', 'no', 'ms', 'o', 'sus', 'lo', 'entre', 'tambin', 'este', 'son', 'the', 'esta', 'pero', 'aos', 'le', 'sobre', 'dos', 'ser', 'ha', 'desde', 'hasta', 'parte', 'durante', 'ciudad', 'donde', 'ao', 'sin', 'era', 'cuando', 'est', 'e', 'gran', 'of', 'despus', 'san', 's', 'tiene', 'ya', 'nombre', 'as']


## Vecinos más cercanos

Probamos con palabras comunes en español.

Si alguna no está en el vocabulario (`min_count` o falta de acento en el corpus), lo verás como OOV (out of vocabulary).


In [23]:
def mostrar_similares(modelo, palabras, titulo, topn=7):
    print(f"\n===== {titulo} =====\n")
    for w in palabras:
        if w in modelo.wv:
            print(f"Más similares a '{w}':")
            for vecina, score in modelo.wv.most_similar(w, topn=topn):
                print(f"  {vecina:15s}  {score:.4f}")
            print()
        else:
            print(f"'{w}' no está en el vocabulario\n")

In [24]:
objetivo = ["rey", "reina", "hombre", "mujer", "madrid", "barcelona"]
mostrar_similares(es_cbow, objetivo, "CBOW")


===== CBOW =====

Más similares a 'rey':
  monarca          0.8431
  prncipe          0.8060
  emperador        0.7932
  trono            0.7711
  sultn            0.7506
  pretendiente     0.7259
  faran            0.7248

Más similares a 'reina':
  infanta          0.7775
  princesa         0.7751
  emperatriz       0.7548
  sibila           0.6905
  zarina           0.6686
  isabel           0.6617
  coronacin        0.6475

Más similares a 'hombre':
  muchacho         0.7505
  demonio          0.7296
  nio              0.7117
  genio            0.7043
  delincuente      0.6956
  vampiro          0.6904
  esclavo          0.6821

Más similares a 'mujer':
  muchacha         0.8088
  nia              0.8042
  prostituta       0.7954
  chica            0.7532
  sirvienta        0.7509
  jovencita        0.7422
  novia            0.7410

Más similares a 'madrid':
  sevilla          0.8530
  valladolid       0.8326
  barcelona        0.8300
  zaragoza         0.8252
  valencia         0

In [25]:
mostrar_similares(es_sg, objetivo, "Skip-gram")


===== Skip-gram =====

Más similares a 'rey':
  monarca          0.8053
  bhumibol         0.7860
  adulyadej        0.7687
  prncipe          0.7662
  astiages         0.7541
  odrisio          0.7538
  pretendiente     0.7527

Más similares a 'reina':
  princesa         0.8040
  isabel           0.7913
  battenberg       0.7910
  infanta          0.7815
  consorte         0.7601
  thyra            0.7467
  tatarabuela      0.7395

Más similares a 'hombre':
  muchacho         0.7583
  andrgino         0.7574
  lujurioso        0.7557
  luntico          0.7538
  depravado        0.7518
  malhechor        0.7419
  ambiciona        0.7413

Más similares a 'mujer':
  jovencita        0.7980
  muchacha         0.7961
  nia              0.7827
  amargada         0.7676
  anciana          0.7647
  solterona        0.7521
  clienta          0.7490

Más similares a 'madrid':
  sevilla          0.8634
  valladolid       0.8366
  zaragoza         0.8230
  gijn             0.8151
  valencia     

## Similitudes entre pares de palabras

El coseno mide qué tan parecidos son dos vectores. Valores cerca de 1 = muy parecidos.


In [26]:
pares = [("rey", "reina"), ("hombre", "mujer"), ("paz", "guerra"), ("madrid", "barcelona")]

print("Similitudes (CBOW):\n")
for a, b in pares:
    if a in es_cbow.wv and b in es_cbow.wv:
        print(f"  sim({a}, {b}) = {es_cbow.wv.similarity(a, b):.4f}")
    else:
        print(f"  Falta {a} o {b}")

Similitudes (CBOW):

  sim(rey, reina) = 0.5596
  sim(hombre, mujer) = 0.4864
  sim(paz, guerra) = 0.4674
  sim(madrid, barcelona) = 0.8300


## Analogías

La analogía clásica: `rey - hombre + mujer ≈ reina`.


In [27]:
def mostrar_analogia(modelo, positivos, negativos, titulo, topn=6):
    print(f"\n=== {titulo} ===\n")
    print(f"Positivos: {positivos}")
    print(f"Negativos: {negativos}\n")
    try:
        for w, s in modelo.wv.most_similar(positive=positivos, negative=negativos, topn=topn):
            print(f"  {w:15s}  {s:.4f}")
    except KeyError as e:
        print(f"Palabra fuera del vocabulario: {e}")

In [28]:
mostrar_analogia(es_cbow, ["rey", "mujer"], ["hombre"], "CBOW: rey - hombre + mujer")


=== CBOW: rey - hombre + mujer ===

Positivos: ['rey', 'mujer']
Negativos: ['hombre']

  emperatriz       0.7129
  reina            0.7096
  princesa         0.6951
  consorte         0.6785
  esposa           0.6701
  infanta          0.6523


In [29]:
mostrar_analogia(es_sg, ["rey", "mujer"], ["hombre"], "Skip-gram: rey - hombre + mujer")


=== Skip-gram: rey - hombre + mujer ===

Positivos: ['rey', 'mujer']
Negativos: ['hombre']

  reina            0.7449
  battenberg       0.7419
  princesa         0.7321
  melisenda        0.7236
  bisabuela        0.7203
  primognita       0.7176


In [30]:
mostrar_analogia(es_cbow, ["padre", "mujer"], ["hombre"], "CBOW: padre - hombre + mujer")


=== CBOW: padre - hombre + mujer ===

Positivos: ['padre', 'mujer']
Negativos: ['hombre']

  esposa           0.8197
  madre            0.8123
  abuela           0.7947
  cuada            0.7579
  sobrina          0.7376
  hermana          0.7374


## Ver el embedding crudo de una palabra

Hasta ahora hemos visto vecinos, sumas y restas. Pero **¿cómo se ve el vector en sí?**

Cada palabra en el modelo es un array de números (100 dimensiones en este caso). Veamos el vector de `"rey"` completo.


In [31]:
if "rey" in es_sg.wv:
    vector_rey = es_sg.wv["rey"]
    print(f"Vector de 'rey' (Skip-gram):")
    print(f"Shape: {vector_rey.shape}")
    print(vector_rey)

Vector de 'rey' (Skip-gram):
Shape: (100,)
[-0.17431723  0.17695974 -0.22158432  0.9064956   0.1636266  -0.03913365
 -0.14322278  0.543734    0.7884705   0.12929924  0.33418444  0.4580398
 -0.79321694  0.15284343  0.13384807 -0.23064709 -0.25726476  0.5414798
 -0.31232226  0.30378333  0.09303294  0.1602298  -0.11883362 -0.08688784
 -0.14332324  0.1910847  -0.14304951 -0.26648098  0.29175195  0.10608851
  0.3937983   0.5692862  -0.69360197 -0.00093723 -0.15208437  0.01070399
  0.17564963 -0.4115729  -0.26252314  0.43741176  0.09061284 -0.105495
 -0.30532622  0.16020691 -0.10803933 -0.25088486  0.01092484  0.07980822
 -0.4655458  -0.5144038   0.36907184  0.20841174 -0.17519927  0.25232476
 -0.03373617 -0.43239808  0.11045196 -0.06712958  0.27106088 -0.24281392
  0.171412    0.00546465  0.75613666  0.09904859  0.30021107  0.39599943
  0.6578689   0.22623056  0.46107972 -0.2784663  -0.06966672 -0.55732656
  0.19212042 -0.39166772  0.15502921  0.15186402 -0.07336818 -0.04778451
 -0.6085979 

Esos 100 números **son la representación** de `"rey"` que el modelo aprendió.

Cuando haces `rey - hombre`, estás restando 100 números menos 100 números. Cuando buscas vecinos, comparas esos 100 números con los 100 números de todas las demás palabras usando coseno.

Los números individuales **no tienen significado directo** (no es que "la posición 5 = género" o algo así). El significado está en la **geometría**: palabras con vectores parecidos (coseno alto) tienen significados parecidos.


## Suma explícita: rey + reina

Sumamos los vectores directamente y buscamos vecinos del resultado.


In [32]:
if "rey" in es_sg.wv and "reina" in es_sg.wv:
    v = es_sg.wv["rey"] + es_sg.wv["reina"]
    print("Suma: rey + reina")
    print("\nVecinos más cercanos:")
    for w, s in vecinos_de_vector(es_sg, v, topn=8, excluir=["rey", "reina"]):
        print(f"  {w:15s}  {s:.4f}")

Suma: rey + reina

Vecinos más cercanos:
  battenberg       0.7918
  plantagenet      0.7727
  bhumibol         0.7675
  consorte         0.7673
  prncipe          0.7650
  pretendiente     0.7627
  haakon           0.7597
  princesa         0.7570


## Suma aleatoria del corpus grande

Tomemos dos palabras al azar (sin stopwords) y veamos qué da su suma.


In [33]:
vocab_corpus_filtrado = [w for w in vocab_corpus if w not in STOP and len(w) > 3]
p1, p2 = random.sample(vocab_corpus_filtrado, 2)
v_random = es_sg.wv[p1] + es_sg.wv[p2]

print(f"Suma aleatoria: {p1} + {p2}")
print("\nPalabras más cercanas:")
for w, s in vecinos_de_vector(es_sg, v_random, topn=8, excluir=[p1, p2]):
    print(f"  {w:15s}  {s:.4f}")

Suma aleatoria: materializadas + frum

Palabras más cercanas:
  portaventura     0.7282
  cccb             0.7142
  patrocinan       0.7102
  estadi           0.7049
  cairota          0.7011
  aramn            0.6983
  mibc             0.6928
  ifema            0.6914


## Otra suma aleatoria


In [34]:
p1, p2 = random.sample(vocab_corpus_filtrado, 2)
v_random = es_sg.wv[p1] + es_sg.wv[p2]

print(f"Suma aleatoria: {p1} + {p2}")
print("\nPalabras más cercanas:")
for w, s in vecinos_de_vector(es_sg, v_random, topn=8, excluir=[p1, p2]):
    print(f"  {w:15s}  {s:.4f}")

Suma aleatoria: minnesota + quarry

Palabras más cercanas:
  kansas           0.8765
  oklahoma         0.8748
  oregon           0.8567
  ohio             0.8555
  utah             0.8452
  wisconsin        0.8447
  alabama          0.8434
  biloxi           0.8371


## Resta aleatoria


In [35]:
p1, p2 = random.sample(vocab_corpus_filtrado, 2)
v_random = es_sg.wv[p1] - es_sg.wv[p2]

print(f"Resta aleatoria: {p1} - {p2}")
print("\nPalabras más cercanas:")
for w, s in vecinos_de_vector(es_sg, v_random, topn=8, excluir=[p1, p2]):
    print(f"  {w:15s}  {s:.4f}")

Resta aleatoria: tsutomu - estatuaria

Palabras más cercanas:
  inukai           0.5551
  yasuo            0.5399
  kiyoshi          0.5368
  hiroshi          0.5325
  masatoshi        0.5276
  tetsu            0.5275
  sasaki           0.5253
  saito            0.5224


## PCA 3D del corpus grande


In [36]:
# No metemos TODO el vocabulario (miles): el gráfico sería ilegible.
# Tomamos las más frecuentes del corpus (sin stopwords), hasta MAX_PALABRAS.
MAX_PALABRAS_PCA = 80
freq_es = Counter(w for o in oraciones_es for w in o)
palabras_plot_es = [
    w for w, _ in freq_es.most_common(500)
    if w not in STOP and w in es_sg.wv and len(w) > 2
][:MAX_PALABRAS_PCA]

coords_es = PCA(n_components=3, random_state=42).fit_transform(
    np.array([es_sg.wv[w] for w in palabras_plot_es])
)
print(f"Palabras en el gráfico: {len(palabras_plot_es)}")

fig2 = go.Figure()
fig2.add_trace(go.Scatter3d(
    x=coords_es[:, 0], y=coords_es[:, 1], z=coords_es[:, 2],
    mode="markers+text",
    text=palabras_plot_es,
    textposition="top center",
    textfont=dict(size=7),
    marker=dict(size=5, color="#16A34A", opacity=0.85),
))
fig2.update_layout(
    title=f"Corpus español (Skip-gram) — PCA 3D — top {len(palabras_plot_es)} palabras frecuentes",
    scene=dict(xaxis_title="PC1", yaxis_title="PC2", zaxis_title="PC3"),
    width=950, height=750,
)
fig2.show()

Palabras en el gráfico: 80


## Exportar embeddings para [TensorFlow Embedding Projector](https://projector.tensorflow.org/)

El [projector](https://projector.tensorflow.org/) espera dos archivos TSV:

1. **Vectores** — una fila por palabra; valores separados por tabulador (`\t`).
2. **Metadatos** — una etiqueta por fila (la palabra), en el mismo orden.

La celda siguiente exporta el último modelo entrenado (`es_sg`, Skip-gram del corpus) a `embedding_projector/`:

- `corpus_skipgram_vectors.tsv` — top 500 palabras frecuentes (sin stopwords)
- `corpus_skipgram_metadata.tsv` — etiqueta por fila

**Uso:** en [projector.tensorflow.org](https://projector.tensorflow.org/) → *Load data from your computer* → primero `corpus_skipgram_vectors.tsv`, luego `corpus_skipgram_metadata.tsv`. Explora con t-SNE, UMAP o PCA en el panel derecho.

In [37]:
def exportar_embedding_projector(modelo, palabras, prefijo, carpeta=None):
    """Escribe *_vectors.tsv y *_metadata.tsv para projector.tensorflow.org."""
    carpeta = carpeta or (BASE_DIR / "embedding_projector")
    carpeta.mkdir(parents=True, exist_ok=True)
    ruta_vectors = carpeta / f"{prefijo}_vectors.tsv"
    ruta_metadata = carpeta / f"{prefijo}_metadata.tsv"

    exportadas = []
    with open(ruta_vectors, "w", encoding="utf-8") as out_v, open(
        ruta_metadata, "w", encoding="utf-8"
    ) as out_m:
        for word in palabras:
            if word not in modelo.wv:
                continue
            vec = modelo.wv[word]
            out_m.write(word + "\n")
            out_v.write("\t".join(f"{x:.6f}" for x in vec) + "\n")
            exportadas.append(word)

    return ruta_vectors, ruta_metadata, exportadas


# Último modelo entrenado: Skip-gram del corpus (es_sg)
MAX_EXPORT_CORPUS = 500
freq_es_export = Counter(w for o in oraciones_es for w in o)
palabras_corpus = [
    w
    for w, _ in freq_es_export.most_common(2000)
    if w not in STOP and len(w) > 2 and w in es_sg.wv
][:MAX_EXPORT_CORPUS]

ruta_vectors, ruta_metadata, palabras_exportadas = exportar_embedding_projector(
    es_sg, palabras_corpus, "corpus_skipgram"
)

print("Archivos listos para https://projector.tensorflow.org/\n")
print(f"Skip-gram — {len(palabras_exportadas)} palabras")
print(f"  vectores:   {ruta_vectors.resolve()}")
print(f"  metadatos:  {ruta_metadata.resolve()}")
print("\nPaso 1: corpus_skipgram_vectors.tsv")
print("Paso 2: corpus_skipgram_metadata.tsv")

Archivos listos para https://projector.tensorflow.org/

Skip-gram — 500 palabras
  vectores:   /Users/emiliosandoval/Documents/ibero/casos-ia-ibero-diplomado/Modulo 4 - NLP/embedding_projector/corpus_skipgram_vectors.tsv
  metadatos:  /Users/emiliosandoval/Documents/ibero/casos-ia-ibero-diplomado/Modulo 4 - NLP/embedding_projector/corpus_skipgram_metadata.tsv

Paso 1: corpus_skipgram_vectors.tsv
Paso 2: corpus_skipgram_metadata.tsv


# Cierre — qué nos falta y por qué vienen los transformers

| Objetivo | Word2Vec | Siguiente paso |
|----------|----------|----------------|
| Sinónimos, ciudades, analogías | Este notebook + **volumen** | Más datos Kaggle / Hugging Face |
| Entender **tu dominio** a fondo | Parcial | Corpus del dominio o embeddings preentrenados |
| **Generar** texto o comparar PDFs | No | **Transformers** (BERT, GPT) |


## Fuentes de corpus en español

- [Kaggle — 120M Spanish Corpus](https://www.kaggle.com/datasets/rtatman/120-million-word-spanish-corpus/data) (58 archivos)
- [Hugging Face Datasets](https://huggingface.co/datasets) (buscar `spanish`, `es`)
- Modelos preentrenados: fastText `cc.es.300.bin`, Spanish 3B words en Zenodo


## Resumen: The Bitter Lesson

1. **Pocas oraciones** → vecinos ruidosos, poca semántica general.
2. **Muchas oraciones** → relaciones estables (rey↔reina, ciudades, familia).
3. **Limpiar ayuda; el volumen manda.**

Cuando quieras que la IA **lea tu corpus con contexto** o **escriba texto nuevo**, pasamos a transformers. Word2Vec sigue siendo la base para entender qué es un **embedding**.


## Experimentos opcionales

1. Cambia `N_ARCHIVOS` (3 vs `None` = todo el Drive) y compara `most_similar('rey')`.
2. `MAX_LINEAS_POR_ARCHIVO = 80_000` vs `None` — tiempo vs calidad.
3. Sube `EPOCHS` o `VECTOR_SIZE` y vuelve a entrenar.
